# Chapter 7: Homotopy and the Fundamental Group

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 7, printed pp. 183-216, PDF pp. 201-234.

## Chapter Goal

This notebook turns the chapter's definitions and proof moves into inspectable computational objects. By the end, a reader should be able to recognize a homotopy as a continuous map on a product with an interval, distinguish free homotopy from path homotopy with endpoints fixed, build the group law on loop classes, track how base points and continuous maps change fundamental groups, and read the category-theoretic statement that `pi_1` is a functor.

The source chapter introduces the fundamental group as the first serious algebraic invariant of a topological space. It begins with homotopy of maps, sharpens the idea to path homotopy relative to endpoints, forms loop classes at a base point, and proves that path-class multiplication gives a group. It then studies what happens under base-point change, continuous maps, products, retractions, and homotopy equivalences. The chapter closes by framing these constructions in the language of categories and functors.

## Computational Translation Guide

The objects below are not replacements for the topological definitions. They are finite or numeric models that make the definitions easier to inspect.

| Book concept | Computational representation | What to inspect |
| --- | --- | --- |
| A homotopy `H: X x I -> Y` | A family of plotted maps indexed by time `t` plus the square `I x I` | Continuity in both variables, not just pointwise motion |
| Path homotopy relative endpoints | Curves whose endpoints remain fixed for every `t` | The boundary conditions on the left and right edges of the square |
| Path multiplication | Piecewise parameter schedules and concatenated words | Multiplication is strict for representatives but associative only after reparametrization/path class |
| Fundamental group | Reduced loop words and finite graph cycle ranks | Identity, inverse, and associativity scaffolds for loop classes |
| Change of base point | Conjugation by a chosen path between base points | Why a base-point isomorphism depends on the chosen path |
| Retraction and deformation retraction | Radial maps and time-indexed Plotly frames | Fixed subspace, final image, and no crossing of the missing point |
| Induced homomorphism | Matrix and word maps on small algebraic models | Preservation of product, identity, and composition |
| Category and functor language | A directed dependency graph | Which theorem supplies each functor axiom |


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-07-homotopy-and-the-fundamental-group/07-homotopy-and-the-fundamental-group.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-07-homotopy-and-the-fundamental-group/07-homotopy-and-the-fundamental-group.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-07-homotopy-and-the-fundamental-group/07-homotopy-and-the-fundamental-group.ipynb",
  "notebook_title": "Chapter 7: Homotopy and the Fundamental Group",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import display


def discover_book_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "Introduction-to-Topological-Manifolds"]
    for parent in here.parents:
        candidates.append(parent / "Introduction-to-Topological-Manifolds")
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "source_map.json").exists() and (candidate / "utils").exists():
            return candidate.resolve()
    raise RuntimeError("Could not discover Introduction-to-Topological-Manifolds root")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, chapter_artifact_root, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html
from utils.topology import abelianization_vector, cycle_rank_for_graph, inverse_letter, word_reduce
from utils.validation import image_stats, relative

UNIT_KEY = "chapter-07-homotopy-and-the-fundamental-group"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print(f"BOOK_ROOT = {relative(BOOK_ROOT, BOOK_ROOT)}")
print(f"ARTIFACT_ROOT = {relative(ARTIFACT_ROOT, BOOK_ROOT)}")


## Library Routing

The chapter is proof-heavy topology, so the notebook uses a small set of libraries in ways that match the geometry. `matplotlib` is used for durable two-dimensional diagrams where the reader must inspect endpoint constraints, path schedules, and base-point conjugation. `plotly` is used for a deformation retraction because the central idea is a time-indexed motion of many points at once; an HTML slider makes the fixed circle and moving radial points easy to compare. `networkx` is used for proof dependency graphs and finite one-dimensional models, where nodes, edges, components, and cycle rank carry the relevant information. `sympy` is used only for exact matrix checks of functoriality and product behavior. `pandas` records route tables and lab tables as durable artifacts.

## Visual Storyboard

The visual sequence follows the chapter's logical order. First, a homotopy square makes the definition `H(s,t)` visible, including the difference between moving a loop freely and keeping endpoints fixed. Second, a path-product schedule and reduced-word scaffold expose why multiplication becomes a group operation only after passing to path classes. Third, a base-point conjugation diagram records the data behind the change-of-base-point isomorphism. Fourth, a radial deformation-retraction slider models a standard homotopy equivalence from the punctured plane to the circle. Fifth, a finite graph lab estimates loop complexity by cycle rank, giving a concrete warning that retractions and homotopy equivalences can preserve loop information without preserving the exact shape. Sixth, a category/functor dependency graph ties continuous maps, induced homomorphisms, homotopy equivalence, and products into one categorical picture.

Every generated artifact is named by the concept it teaches. The JSON storyboard below is saved under `checks/visual-storyboard.json` so that the final sanity cell can verify the notebook's intended visual coverage.


In [ ]:
storyboard = [
    {
        "concept": "homotopy_square_and_relative_endpoints",
        "representation": "Matplotlib path family plus domain square",
        "library": "matplotlib, numpy",
        "artifact": "figures/path-homotopy-relative-endpoints.png",
        "inspection_target": "Endpoints stay fixed while interior points move; the straight family through the puncture shows why endpoint discipline alone is not enough to prove a loop is null.",
        "validation": "Endpoint coordinates are constant for all sampled times; midpoint hits the marked puncture only in the illustrative straight family.",
    },
    {
        "concept": "path_product_reparametrization",
        "representation": "Piecewise time-allocation diagram and reduced loop words",
        "library": "matplotlib, utils.topology",
        "artifact": "figures/path-product-reparametrization.png",
        "inspection_target": "The two parenthesizations traverse f, g, h in the same order but with different speeds.",
        "validation": "Reduced-word identity, inverse, and associativity scaffolds pass exactly.",
    },
    {
        "concept": "basepoint_change_conjugation",
        "representation": "Path-and-loop diagram at two base points",
        "library": "matplotlib, numpy",
        "artifact": "figures/basepoint-change-conjugation.png",
        "inspection_target": "A loop based at p becomes a loop based at q by traveling back along g, doing the loop, then returning along g.",
        "validation": "Sampled concatenated path starts and ends at q and the symbolic conjugation map preserves products.",
    },
    {
        "concept": "deformation_retraction_punctured_plane",
        "representation": "Plotly slider for H(x,t) = (1-t)x + t x/|x|",
        "library": "plotly, numpy",
        "artifact": "html/deformation-retraction-punctured-plane.html",
        "inspection_target": "All points move radially to S^1, points already on S^1 remain fixed, and no sampled point crosses the origin.",
        "validation": "Final radii equal one within tolerance and minimum sampled radius stays positive.",
    },
    {
        "concept": "finite_loop_rank_lab",
        "representation": "NetworkX graph models and CSV cycle-rank table",
        "library": "networkx, pandas, matplotlib",
        "artifact": "figures/finite-loop-rank-lab.png",
        "inspection_target": "Cycle rank E - V + components predicts the number of independent loops in finite graph models.",
        "validation": "Connected graph ranks are nonnegative and match the saved table.",
    },
    {
        "concept": "functoriality_and_categories",
        "representation": "NetworkX dependency graph plus exact SymPy matrix checks",
        "library": "networkx, sympy, matplotlib",
        "artifact": "figures/category-functor-dependency-graph.png",
        "inspection_target": "Propositions about induced homomorphisms supply the functor laws; homotopy equivalence adds isomorphism invariance.",
        "validation": "Identity, composition, and product projection checks pass exactly.",
    },
]

library_rows = [
    {"concept": item["concept"], "library": item["library"], "artifact": item["artifact"], "why": item["inspection_target"]}
    for item in storyboard
]

storyboard_path = save_json(storyboard, CHECKS / "visual-storyboard.json")
library_table_path = save_csv(library_rows, TABLES / "library-routing-and-storyboard.csv")

display(pd.DataFrame(library_rows))
display_artifact(storyboard_path)
print(relative(storyboard_path, BOOK_ROOT))
print(relative(library_table_path, BOOK_ROOT))


## Homotopy as a Map on a Square

A homotopy is not merely a sequence of unrelated pictures. It is one continuous map on a product `X x I`. For paths, the domain is the square `I x I`, where `s` moves along a path and `t` moves through time. The bottom edge is the starting path, the top edge is the ending path, and path homotopy adds two side-edge conditions: the left edge is the initial point for all time and the right edge is the terminal point for all time.

The figure generated below uses two arcs from `(-1,0)` to `(1,0)`. The straight interpolation between them is a legitimate path homotopy in the whole plane because the endpoints remain fixed and the map varies continuously. If the origin is removed, the displayed straight interpolation is no longer a homotopy in the punctured plane because the midpoint path touches the missing point. This does not prove that the two arcs are not path-homotopic in the punctured plane; that nontrivial result belongs to the next chapter's circle computation. The point here is more local: the codomain matters, and continuity must be checked for the full square.

This example also explains why the chapter first shows that unrestricted homotopy is an equivalence relation, then repeats the argument relative to the endpoints for paths. The proofs have the same shape: reverse the time coordinate for symmetry, and glue two homotopies along the middle time slice for transitivity. The square makes those proof moves visible.


In [ ]:
s = np.linspace(0.0, 1.0, 401)
x = 2.0 * s - 1.0
upper_y = np.sin(np.pi * s)

times = [0.0, 0.25, 0.5, 0.75, 1.0]
colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(times)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ax = axes[0]
for t, color in zip(times, colors):
    y = (1.0 - 2.0 * t) * upper_y
    ax.plot(x, y, color=color, lw=2.4, label=f"t={t:.2f}")
ax.scatter([-1, 1], [0, 0], s=70, color="black", zorder=4, label="fixed endpoints")
ax.scatter([0], [0], s=95, facecolor="white", edgecolor="crimson", linewidth=2.5, zorder=5, label="removed point")
ax.set_aspect("equal", adjustable="box")
ax.set_title("Path family H(s,t) in the plane")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)

ax = axes[1]
ax.set_title("Domain square I x I")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
for t, color in zip(times, colors):
    ax.plot([0, 1], [t, t], color=color, lw=2.0)
ax.plot([0, 0], [0, 1], color="black", lw=3)
ax.plot([1, 1], [0, 1], color="black", lw=3)
ax.plot([0, 1], [0, 0], color="tab:blue", lw=3)
ax.plot([0, 1], [1, 1], color="tab:orange", lw=3)
ax.scatter([0.5], [0.5], s=80, facecolor="white", edgecolor="crimson", linewidth=2)
ax.text(0.03, 0.5, "initial point fixed", rotation=90, va="center")
ax.text(0.91, 0.5, "terminal point fixed", rotation=90, va="center")
ax.text(0.36, -0.09, "starting path")
ax.text(0.38, 1.04, "ending path")
ax.set_xlabel("path parameter s")
ax.set_ylabel("homotopy time t")

homotopy_endpoint_errors = []
for t in np.linspace(0, 1, 21):
    y = (1.0 - 2.0 * t) * upper_y
    homotopy_endpoint_errors.append(abs(y[0]) + abs(y[-1]))

homotopy_checks = {
    "endpoint_error_max": float(max(homotopy_endpoint_errors)),
    "midpoint_hits_removed_point_in_straight_family": bool(abs((1.0 - 2.0 * 0.5) * np.sin(np.pi * 0.5)) < 1e-12),
    "sampled_time_count": len(times),
    "interpretation": "This is a valid path homotopy in R^2; as drawn it is not a homotopy in R^2 minus the origin.",
}

path_homotopy_png = save_matplotlib(fig, FIGURES / "path-homotopy-relative-endpoints.png")
plt.close(fig)
path_homotopy_json = save_json(homotopy_checks, CHECKS / "path-homotopy-relative-endpoints-checks.json")

display_artifact(path_homotopy_png, width=920)
display_artifact(path_homotopy_json)


## From Paths to a Group

The fundamental group is built by taking loops based at a point, identifying loops that are path-homotopic, and multiplying classes by concatenating representatives. The representative-level product is piecewise: follow the first path at double speed, then the second path at double speed. The gluing lemma supplies continuity because the terminal point of the first path equals the initial point of the second.

There is a subtlety that the notebook keeps visible. Products of paths are not literally associative as functions `I -> X`, because `(f*g)*h` and `f*(g*h)` spend different proportions of the unit interval on `f`, `g`, and `h`. They traverse the same path pieces in the same order, and Lemma 7.9 says that reparametrizing a path does not change its path-homotopy class. Associativity therefore holds at the level of path classes. The identity class is the constant loop, and the inverse of a loop is the same path traced backward.

The reduced-word calculation below is a finite algebraic scaffold for these group laws. It is not claiming that every fundamental group is free. It merely gives a small exact setting where identity, inverse, and associativity can be checked without numerical tolerance. That exactness is useful because the topological proofs are also exact: they build explicit homotopies and then pass to equivalence classes.


In [ ]:
def inverse_word(word: list[str]) -> list[str]:
    return [inverse_letter(letter) for letter in reversed(word)]


w1 = ["a", "b", "a^-1"]
w2 = ["a", "b^-1"]
w3 = ["b", "a"]
identity_word: list[str] = []

left_assoc = word_reduce(word_reduce(w1 + w2) + w3)
right_assoc = word_reduce(w1 + word_reduce(w2 + w3))
inverse_cancel = word_reduce(w1 + inverse_word(w1))
identity_cancel = word_reduce(identity_word + w2) == word_reduce(w2 + identity_word) == word_reduce(w2)
conjugated_w1 = word_reduce(inverse_word(["g"]) + w1 + ["g"])
conjugated_w2 = word_reduce(inverse_word(["g"]) + w2 + ["g"])
conjugated_product = word_reduce(inverse_word(["g"]) + word_reduce(w1 + w2) + ["g"])
product_of_conjugates = word_reduce(conjugated_w1 + conjugated_w2)

word_checks = {
    "left_associative_reduction": left_assoc,
    "right_associative_reduction": right_assoc,
    "associativity_scaffold_passed": left_assoc == right_assoc,
    "inverse_cancels_to_identity": inverse_cancel == [],
    "identity_cancels": identity_cancel,
    "conjugation_preserves_product_scaffold": conjugated_product == product_of_conjugates,
    "abelianization_of_w1w2": abelianization_vector(w1 + w2, ["a", "b"]),
}
assert word_checks["associativity_scaffold_passed"]
assert word_checks["inverse_cancels_to_identity"]
assert word_checks["identity_cancels"]
assert word_checks["conjugation_preserves_product_scaffold"]

fig, ax = plt.subplots(figsize=(10, 4.6))
rows = ["(f*g)*h", "f*(g*h)"]
segments = {
    "(f*g)*h": [(0.00, 0.25, "f", "tab:blue"), (0.25, 0.50, "g", "tab:green"), (0.50, 1.00, "h", "tab:orange")],
    "f*(g*h)": [(0.00, 0.50, "f", "tab:blue"), (0.50, 0.75, "g", "tab:green"), (0.75, 1.00, "h", "tab:orange")],
}
for y, row in enumerate(rows):
    for start, end, label, color in segments[row]:
        ax.broken_barh([(start, end - start)], (y - 0.32, 0.64), facecolors=color, alpha=0.82)
        ax.text((start + end) / 2, y, label, ha="center", va="center", color="white", fontsize=13, weight="bold")
ax.set_yticks([0, 1], labels=rows)
ax.set_xlabel("unit interval parameter s")
ax.set_xlim(0, 1)
ax.set_ylim(-0.7, 1.7)
ax.set_title("Same ordered path pieces, different speed schedules")
for breakpoint in [0.25, 0.5, 0.75]:
    ax.axvline(breakpoint, color="black", alpha=0.18, lw=1)
ax.text(0.5, -0.55, "Reparametrization turns this timing difference into a path homotopy of representatives.", ha="center")

product_png = save_matplotlib(fig, FIGURES / "path-product-reparametrization.png")
plt.close(fig)
word_checks_path = save_json(word_checks, CHECKS / "path-class-word-checks.json")

display_artifact(product_png, width=850)
display(pd.DataFrame([word_checks]))
display_artifact(word_checks_path)


## Base Points and Conjugation

A loop has a base point, and the base point is not bookkeeping that can always be ignored. If a path-connected space has two points `p` and `q`, a chosen path `g` from `p` to `q` gives an isomorphism from `pi_1(X,p)` to `pi_1(X,q)`. The construction is conjugation by the travel path: start at `q`, go backward along `g` to `p`, follow the old loop, and return to `q` along `g`.

The diagram below is a visual version of that formula. Its main lesson is dependence on data. The points `p` and `q` alone do not specify the isomorphism; the chosen path between them does. In spaces whose fundamental group is nonabelian, different choices can produce different conjugations. This is why the chapter later introduces pointed spaces and pointed maps when it wants a clean functor `pi_1: Top_* -> Grp`.

This base-point issue also explains a technical point in the homotopy invariance proof. If two maps are homotopic, they need not send the base point to the same point at intermediate time. The homotopy traces a path between the two image base points, and Lemma 7.45 uses exactly the change-of-base-point isomorphism to make the diagram commute.


In [ ]:
def quadratic_bezier(p0: np.ndarray, p1: np.ndarray, p2: np.ndarray, samples: int = 180) -> np.ndarray:
    t = np.linspace(0, 1, samples)[:, None]
    return (1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2


p = np.array([-1.15, 0.0])
q = np.array([1.25, 0.0])
g_path = quadratic_bezier(p, np.array([0.0, 0.78]), q)
loop_theta = np.linspace(0, 2 * np.pi, 240)
loop_p = p + np.column_stack([0.36 * np.cos(loop_theta), 0.27 * np.sin(loop_theta)])
conjugate_path = np.vstack([g_path[::-1], loop_p, g_path])

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(g_path[:, 0], g_path[:, 1], color="tab:purple", lw=3, label="g: p -> q")
ax.plot(g_path[::-1, 0], g_path[::-1, 1] - 0.08, color="tab:purple", lw=1.8, ls="--", label="reverse of g")
ax.plot(loop_p[:, 0], loop_p[:, 1], color="tab:blue", lw=2.6, label="loop f at p")
ax.plot(conjugate_path[:, 0], conjugate_path[:, 1] - 0.52, color="tab:red", lw=2.2, alpha=0.82, label="based at q: reverse(g) * f * g")
ax.scatter([p[0], q[0]], [p[1], q[1]], s=90, color="black", zorder=5)
ax.text(p[0] - 0.08, p[1] + 0.15, "p", fontsize=14, weight="bold")
ax.text(q[0] + 0.04, q[1] + 0.15, "q", fontsize=14, weight="bold")
ax.annotate("", xy=g_path[110], xytext=g_path[80], arrowprops={"arrowstyle": "->", "color": "tab:purple", "lw": 2})
ax.annotate("", xy=loop_p[80], xytext=loop_p[55], arrowprops={"arrowstyle": "->", "color": "tab:blue", "lw": 2})
ax.set_aspect("equal", adjustable="box")
ax.set_title("Change of base point as conjugation by a path")
ax.set_xlabel("model coordinate")
ax.set_ylabel("model coordinate")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)

basepoint_checks = {
    "g_starts_at_p": bool(np.allclose(g_path[0], p)),
    "g_ends_at_q": bool(np.allclose(g_path[-1], q)),
    "conjugate_loop_starts_at_q": bool(np.allclose(conjugate_path[0], q)),
    "conjugate_loop_ends_at_q": bool(np.allclose(conjugate_path[-1], q)),
    "symbolic_formula": "Phi_g([f]) = [reverse(g) * f * g]",
}
assert all(value for key, value in basepoint_checks.items() if isinstance(value, bool))

basepoint_png = save_matplotlib(fig, FIGURES / "basepoint-change-conjugation.png")
plt.close(fig)
basepoint_json = save_json(basepoint_checks, CHECKS / "basepoint-change-conjugation-checks.json")

display_artifact(basepoint_png, width=760)
display_artifact(basepoint_json)


## Retractions, Deformation Retractions, and Homotopy Invariance

A retraction `r: X -> A` fixes the subspace `A` pointwise. A deformation retraction strengthens this by requiring the identity map on `X` to be homotopic to the inclusion followed by `r`. A strong deformation retraction keeps every point of `A` fixed throughout the homotopy. This distinction matters because the chapter uses retractions to get injective maps on fundamental groups, then uses deformation retractions and homotopy equivalences to get isomorphisms.

The radial map from the punctured plane to the unit circle is the standard model. For `x != 0`, the formula `H(x,t) = (1-t)x + t x/|x|` moves each point along the radial segment to the circle. Points already on the circle remain fixed for every `t`, and no point crosses the origin. This is exactly the behavior a strong deformation retraction should have.

The Plotly artifact is useful here because the concept is time-dependent. Drag the slider from `t=0` to `t=1`. The ambient punctured plane has many radii, but the final frame lies on `S^1`. Since the inclusion of the circle and the radial retraction are homotopy inverses, the chapter's homotopy invariance theorem says they induce isomorphic fundamental groups.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 25, endpoint=False)
radii = np.array([0.45, 0.75, 1.0, 1.45, 1.9])
points = np.array([[r * np.cos(a), r * np.sin(a)] for r in radii for a in theta])
unit_points = points / np.linalg.norm(points, axis=1)[:, None]
frame_times = np.linspace(0.0, 1.0, 9)

circle_theta = np.linspace(0, 2 * np.pi, 300)
circle_x = np.cos(circle_theta)
circle_y = np.sin(circle_theta)


def retracted_points(t: float) -> np.ndarray:
    return (1.0 - t) * points + t * unit_points


frames = []
for t in frame_times:
    moved = retracted_points(float(t))
    frames.append(go.Frame(
        name=f"{t:.2f}",
        data=[
            go.Scatter(x=circle_x, y=circle_y, mode="lines", line={"color": "black", "width": 2}, name="S^1 fixed"),
            go.Scatter(x=moved[:, 0], y=moved[:, 1], mode="markers", marker={"size": 6, "color": np.linalg.norm(moved, axis=1), "colorscale": "Viridis", "showscale": True, "colorbar": {"title": "radius"}}, name="H(x,t)"),
        ],
        layout=go.Layout(title_text=f"Strong deformation retraction of R^2 minus 0 onto S^1, t={t:.2f}"),
    ))

initial = retracted_points(0.0)
fig = go.Figure(
    data=[
        go.Scatter(x=circle_x, y=circle_y, mode="lines", line={"color": "black", "width": 2}, name="S^1 fixed"),
        go.Scatter(x=initial[:, 0], y=initial[:, 1], mode="markers", marker={"size": 6, "color": np.linalg.norm(initial, axis=1), "colorscale": "Viridis", "showscale": True, "colorbar": {"title": "radius"}}, name="H(x,t)"),
    ],
    frames=frames,
)
fig.update_layout(
    title="Strong deformation retraction of the punctured plane onto S^1",
    xaxis={"range": [-2.1, 2.1], "scaleanchor": "y", "title": "x"},
    yaxis={"range": [-2.1, 2.1], "title": "y"},
    width=760,
    height=650,
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "t = "},
        "steps": [{"label": f"{t:.2f}", "method": "animate", "args": [[f"{t:.2f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]} for t in frame_times],
    }],
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 450, "redraw": True}, "fromcurrent": True}]},
            {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
        ],
    }],
)

all_moved = np.vstack([retracted_points(float(t)) for t in frame_times])
unit_fixed_error = max(np.linalg.norm((1.0 - t) * unit_points + t * unit_points - unit_points, axis=1).max() for t in frame_times)
final_radius_error = float(np.max(np.abs(np.linalg.norm(retracted_points(1.0), axis=1) - 1.0)))
min_radius = float(np.min(np.linalg.norm(all_moved, axis=1)))

deformation_checks = {
    "sampled_point_count": int(points.shape[0]),
    "frame_count": int(len(frame_times)),
    "minimum_sampled_radius_during_homotopy": min_radius,
    "final_radius_error_max": final_radius_error,
    "unit_circle_fixed_error_max": float(unit_fixed_error),
    "strong_deformation_retraction_checks_passed": bool(min_radius > 0.0 and final_radius_error < 1e-12 and unit_fixed_error < 1e-12),
}
assert deformation_checks["strong_deformation_retraction_checks_passed"]

deformation_html = save_plotly_html(fig, HTML / "deformation-retraction-punctured-plane.html")
deformation_json = save_json(deformation_checks, CHECKS / "deformation-retraction-punctured-plane-checks.json")

display_artifact(deformation_html, width=820, height=700)
display_artifact(deformation_json)


## Spheres, Manifolds, and a Finite Loop-Rank Lab

The chapter proves that `S^n` is simply connected for `n >= 2`. The proof uses two ideas that are computationally suggestive. First, the Lebesgue number lemma lets us subdivide a compact domain finely enough that each path segment lands in a controlled coordinate ball. Second, in dimension at least two, removing one point from a coordinate ball does not disconnect it, so path segments can be nudged away from a chosen point. Once a loop in `S^n` misses a point, stereographic projection moves it into `R^n`, where it is null-homotopic.

The countability theorem for fundamental groups of manifolds has a similar finite-subdivision flavor. A loop is chopped into small pieces, each piece is replaced by one of countably many chosen paths inside coordinate balls, and the whole loop class is represented by a finite product from a countable list. The argument is not an algorithm for arbitrary manifolds, but it does explain why local charts plus compactness can turn a continuous path into finitely many controlled combinatorial choices.

The lab below uses finite graphs as a deliberately simpler setting. For a connected finite graph, the number `E - V + 1` is the rank of its cycle space and, geometrically, the number of independent loop generators for the graph. This previews later computations without importing later theorems into the chapter. The point is to practice the invariant habit: replace a drawn space by a model whose loop information can be counted, then ask which maps preserve that information.


In [ ]:
def graph_summary(name: str, graph: nx.Graph) -> dict[str, int | str]:
    components = nx.number_connected_components(graph)
    vertices = graph.number_of_nodes()
    edges = graph.number_of_edges()
    rank = cycle_rank_for_graph(vertices, edges, components)
    return {"space_model": name, "vertices": vertices, "edges": edges, "components": components, "cycle_rank": rank}


circle = nx.cycle_graph(8)
figure_eight = nx.Graph()
figure_eight.add_edges_from([
    ("v0", "a1"), ("a1", "a2"), ("a2", "v0"),
    ("v0", "b1"), ("b1", "b2"), ("b2", "v0"),
])
theta_graph = nx.Graph()
theta_graph.add_edges_from([
    ("p", "m1"), ("m1", "q"),
    ("p", "m2"), ("m2", "q"),
    ("p", "m3"), ("m3", "q"),
])
tree = nx.path_graph(6)

graphs = {
    "circle graph": circle,
    "figure-eight model": figure_eight,
    "theta model": theta_graph,
    "tree model": tree,
}

positions = {
    "circle graph": nx.circular_layout(circle),
    "figure-eight model": {
        "v0": np.array([0.0, 0.0]), "a1": np.array([-0.9, 0.7]), "a2": np.array([-0.9, -0.7]),
        "b1": np.array([0.9, 0.7]), "b2": np.array([0.9, -0.7]),
    },
    "theta model": {"p": np.array([-1.0, 0.0]), "q": np.array([1.0, 0.0]), "m1": np.array([0.0, 0.8]), "m2": np.array([0.0, 0.0]), "m3": np.array([0.0, -0.8])},
    "tree model": {i: np.array([i, 0.25 * ((-1) ** i)]) for i in tree.nodes},
}

rows = [graph_summary(name, graph) for name, graph in graphs.items()]
rank_df = pd.DataFrame(rows)
rank_csv = save_csv(rank_df.to_dict(orient="records"), TABLES / "finite-loop-rank-lab.csv")

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, graph) in zip(axes.flat, graphs.items()):
    pos = positions[name]
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#4a5568", width=2.2)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_size=240, node_color="#edf2f7", edgecolors="#2d3748", linewidths=1.4)
    summary = graph_summary(name, graph)
    ax.set_title(f"{name}: rank {summary['cycle_rank']}")
    ax.set_axis_off()
fig.suptitle("Finite graph loop-rank lab: cycle_rank = E - V + components", y=0.98)

rank_checks = {
    "rows": rows,
    "all_ranks_nonnegative": bool(all(row["cycle_rank"] >= 0 for row in rows)),
    "figure_eight_rank_is_two": bool(rank_df.loc[rank_df["space_model"] == "figure-eight model", "cycle_rank"].item() == 2),
    "theta_rank_is_two": bool(rank_df.loc[rank_df["space_model"] == "theta model", "cycle_rank"].item() == 2),
    "tree_rank_is_zero": bool(rank_df.loc[rank_df["space_model"] == "tree model", "cycle_rank"].item() == 0),
}
assert rank_checks["all_ranks_nonnegative"]
assert rank_checks["figure_eight_rank_is_two"]
assert rank_checks["theta_rank_is_two"]
assert rank_checks["tree_rank_is_zero"]

rank_png = save_matplotlib(fig, FIGURES / "finite-loop-rank-lab.png")
plt.close(fig)
rank_json = save_json(rank_checks, CHECKS / "finite-loop-rank-lab-checks.json")

display(rank_df)
display_artifact(rank_png, width=820)
display_artifact(rank_json)
print(relative(rank_csv, BOOK_ROOT))


## Induced Homomorphisms, Products, and Functors

A continuous map `phi: X -> Y` sends a loop in `X` based at `p` to a loop in `Y` based at `phi(p)` by composition. Passing to path classes gives an induced homomorphism `phi_*: pi_1(X,p) -> pi_1(Y,phi(p))`. The proof that this is well-defined uses path homotopy preservation by composition. The proof that it is a homomorphism uses the literal equality `phi o (f*g) = (phi o f) * (phi o g)`.

The induced maps satisfy two functor laws: identities induce identities, and a composite of continuous maps induces the composite of the induced homomorphisms. In category language, `pi_1` is a covariant functor from pointed topological spaces to groups. Homotopy equivalence goes further: if `phi` has a homotopy inverse, then `phi_*` is an isomorphism, with base-point corrections supplied by the path traced by the homotopy.

Product spaces give a second inspectable functorial pattern. A loop in a product is the same data as a tuple of component loops, so `pi_1(X_1 x ... x X_n)` maps isomorphically to the product of the fundamental groups of the factors. The exact matrix checks below model this with integer lattices: `pi_1(T^2)` behaves like `Z^2`, projections behave like coordinate projections, and composition of induced homomorphisms behaves like matrix multiplication.


In [ ]:
# Exact algebraic model for induced maps on pi_1(T^2) ~= Z^2.
z1, z2 = sp.symbols("z1 z2", integer=True)
v = sp.Matrix([z1, z2])
I2 = sp.eye(2)
A = sp.Matrix([[2, 1], [0, 1]])
B = sp.Matrix([[1, 0], [3, 1]])
P1 = sp.Matrix([[1, 0]])
P2 = sp.Matrix([[0, 1]])

identity_check = sp.simplify(I2 * v - v) == sp.zeros(2, 1)
composition_check = sp.simplify((B * A) * v - B * (A * v)) == sp.zeros(2, 1)
projection_product_check = (P1 * v)[0] == z1 and (P2 * v)[0] == z2
homomorphism_addition_check = A * (sp.Matrix([1, 2]) + sp.Matrix([3, -1])) == A * sp.Matrix([1, 2]) + A * sp.Matrix([3, -1])

functor_checks = {
    "identity_induces_identity_matrix": bool(identity_check),
    "composition_matches_matrix_product": bool(composition_check),
    "torus_product_projections_pick_coordinates": bool(projection_product_check),
    "integer_matrix_preserves_addition": bool(homomorphism_addition_check),
    "example_A": [[int(A[i, j]) for j in range(A.shape[1])] for i in range(A.shape[0])],
    "example_B": [[int(B[i, j]) for j in range(B.shape[1])] for i in range(B.shape[0])],
}
assert functor_checks["identity_induces_identity_matrix"]
assert functor_checks["composition_matches_matrix_product"]
assert functor_checks["torus_product_projections_pick_coordinates"]
assert functor_checks["integer_matrix_preserves_addition"]

G = nx.DiGraph()
G.add_edges_from([
    ("Top_*", "pointed maps"),
    ("pointed maps", "path homotopy preserved"),
    ("path homotopy preserved", "induced homomorphism phi_*"),
    ("induced homomorphism phi_*", "identity law"),
    ("induced homomorphism phi_*", "composition law"),
    ("identity law", "pi_1 functor to Grp"),
    ("composition law", "pi_1 functor to Grp"),
    ("homotopic maps", "base-point path correction"),
    ("base-point path correction", "homotopy invariance"),
    ("homotopy equivalence", "homotopy invariance"),
    ("homotopy invariance", "isomorphic fundamental groups"),
    ("product projections", "pi_1 product isomorphism"),
    ("pi_1 functor to Grp", "pi_1 product isomorphism"),
    ("categories", "functors preserve isomorphisms"),
    ("pi_1 functor to Grp", "functors preserve isomorphisms"),
])

pos = nx.spring_layout(G, seed=7, k=1.0)
fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, edge_color="#718096", width=1.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1700, node_color="#e6fffa", edgecolors="#234e52", linewidths=1.4)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5)
ax.set_title("Proof dependency graph for induced maps, functoriality, and homotopy invariance")
ax.set_axis_off()

functor_png = save_matplotlib(fig, FIGURES / "category-functor-dependency-graph.png")
plt.close(fig)
functor_json = save_json(functor_checks, CHECKS / "functoriality-and-products-checks.json")

display(pd.DataFrame([functor_checks]))
display_artifact(functor_png, width=900)
display_artifact(functor_json)


## Circle Representatives, the Square Lemma, and Higher Homotopy

A loop `f: I -> X` has a circle representative because the endpoints of `I` are identified to form `S^1`. This lets the chapter translate between based loops and maps from the circle. A loop is null-homotopic exactly when its circle representative is freely homotopic to a constant map, equivalently when that representative extends over the disk. The disk extension condition is a powerful geometric test: a loop that bounds a continuous disk in the space has no fundamental-group obstruction.

The square lemma is the chapter's compact proof tool for boundary paths. Given a continuous map from the square into a space, the path along the bottom followed by the right side is path-homotopic to the path along the left side followed by the top. This is the same intuition as sliding a route across a filled square. It is used in the proof that homotopic maps induce the same homomorphism after correcting the base point.

Higher homotopy groups generalize the circle-representative viewpoint by replacing `S^1` with `S^n`. The notebook does not compute them, matching the chapter's role: it introduces the notation and the idea that `pi_n` measures `n`-dimensional holes. The important bridge for this chapter is conceptual. The fundamental group is the first case where paths, endpoint constraints, group structure, and functoriality all become visible at once.


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.plot([0, 1], [0, 0], color="tab:blue", lw=3)
ax.plot([1, 1], [0, 1], color="tab:green", lw=3)
ax.plot([0, 1], [1, 1], color="tab:orange", lw=3)
ax.plot([0, 0], [0, 1], color="tab:red", lw=3)
ax.annotate("f", xy=(0.5, -0.05), ha="center", va="top", color="tab:blue", fontsize=14)
ax.annotate("g", xy=(1.06, 0.5), ha="left", va="center", color="tab:green", fontsize=14)
ax.annotate("k", xy=(0.5, 1.06), ha="center", va="bottom", color="tab:orange", fontsize=14)
ax.annotate("h", xy=(-0.06, 0.5), ha="right", va="center", color="tab:red", fontsize=14)
for start, end, color in [((0.18, 0), (0.36, 0), "tab:blue"), ((1, 0.18), (1, 0.36), "tab:green"), ((0.82, 1), (0.64, 1), "tab:orange"), ((0, 0.82), (0, 0.64), "tab:red")]:
    ax.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "->", "lw": 2, "color": color})
ax.text(0.5, 0.5, "F: I x I -> X", ha="center", va="center", fontsize=13, bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#4a5568"})
ax.set_xlim(-0.18, 1.18)
ax.set_ylim(-0.18, 1.18)
ax.set_aspect("equal")
ax.set_title("Square lemma boundary paths: f*g is homotopic to h*k")
ax.set_axis_off()

square_checks = {
    "bottom_then_right_start": [0.0, 0.0],
    "bottom_then_right_end": [1.0, 1.0],
    "left_then_top_start": [0.0, 0.0],
    "left_then_top_end": [1.0, 1.0],
    "endpoint_pairs_match": True,
}
assert square_checks["endpoint_pairs_match"]

square_png = save_matplotlib(fig, FIGURES / "square-lemma-boundary-paths.png")
plt.close(fig)
square_json = save_json(square_checks, CHECKS / "square-lemma-boundary-paths-checks.json")

display_artifact(square_png, width=560)
display_artifact(square_json)


## Applied Lab: Retraction Radar

Use the artifacts above as a small decision workflow for drawn spaces.

1. Mark the base point and decide whether the map you care about preserves it. If not, record the path that changes base points.
2. Look for a visible retract or deformation retract. A plain retract gives injectivity for the included subspace; a deformation retract gives homotopy equivalence and therefore an isomorphism on fundamental groups.
3. If the space can be modeled by a finite graph without losing the loop information you are studying, compute `E - V + components` as a loop-rank scaffold.
4. For maps between product-like spaces, model induced maps by matrices or coordinate projections and check identity and composition before making a topological claim.
5. Translate the result into category language only after the base points, induced maps, and homotopies have been specified.

The lab deliberately separates evidence from theorem. A graph rank, a reduced word, or a matrix check can make a theorem plausible and help catch a wrong map, but the topological proof still requires the hypotheses in the chapter: continuity, endpoint conditions, gluing, homotopy relative to a subset, or a specified homotopy inverse.


In [ ]:
artifact_paths = [
    CHECKS / "visual-storyboard.json",
    TABLES / "library-routing-and-storyboard.csv",
    FIGURES / "path-homotopy-relative-endpoints.png",
    CHECKS / "path-homotopy-relative-endpoints-checks.json",
    FIGURES / "path-product-reparametrization.png",
    CHECKS / "path-class-word-checks.json",
    FIGURES / "basepoint-change-conjugation.png",
    CHECKS / "basepoint-change-conjugation-checks.json",
    HTML / "deformation-retraction-punctured-plane.html",
    CHECKS / "deformation-retraction-punctured-plane-checks.json",
    TABLES / "finite-loop-rank-lab.csv",
    FIGURES / "finite-loop-rank-lab.png",
    CHECKS / "finite-loop-rank-lab-checks.json",
    FIGURES / "category-functor-dependency-graph.png",
    CHECKS / "functoriality-and-products-checks.json",
    FIGURES / "square-lemma-boundary-paths.png",
    CHECKS / "square-lemma-boundary-paths-checks.json",
]

assert_artifacts(artifact_paths, min_bytes=64)

with (CHECKS / "visual-storyboard.json").open(encoding="utf-8") as handle:
    loaded_storyboard = json.load(handle)
assert len(loaded_storyboard) >= 6

with (CHECKS / "path-class-word-checks.json").open(encoding="utf-8") as handle:
    loaded_word_checks = json.load(handle)
assert loaded_word_checks["associativity_scaffold_passed"]
assert loaded_word_checks["inverse_cancels_to_identity"]
assert loaded_word_checks["conjugation_preserves_product_scaffold"]

with (CHECKS / "deformation-retraction-punctured-plane-checks.json").open(encoding="utf-8") as handle:
    loaded_deformation = json.load(handle)
assert loaded_deformation["strong_deformation_retraction_checks_passed"]

with (CHECKS / "functoriality-and-products-checks.json").open(encoding="utf-8") as handle:
    loaded_functor = json.load(handle)
assert loaded_functor["identity_induces_identity_matrix"]
assert loaded_functor["composition_matches_matrix_product"]
assert loaded_functor["torus_product_projections_pick_coordinates"]

png_stats = [image_stats(path) for path in artifact_paths if path.suffix.lower() == ".png"]
assert all(item["width"] >= 500 and item["height"] >= 350 for item in png_stats)
assert all(item["max_channel_stddev"] > 3.0 for item in png_stats)

rank_table = pd.read_csv(TABLES / "finite-loop-rank-lab.csv")
assert {"circle graph", "figure-eight model", "theta model", "tree model"}.issubset(set(rank_table["space_model"]))

final_sanity = {
    "artifact_count": len(artifact_paths),
    "png_count": len(png_stats),
    "json_check_count": len([path for path in artifact_paths if path.suffix.lower() == ".json"]),
    "storyboard_items": len(loaded_storyboard),
    "all_artifacts_relative_to_book": [relative(path, BOOK_ROOT) for path in artifact_paths],
    "status": "passed",
}

save_json(final_sanity, CHECKS / "final-sanity-checks.json")
display(pd.DataFrame(png_stats))
display(final_sanity)


## Takeaways

The fundamental group packages loops based at a point into an algebraic invariant. The base point matters because loop multiplication starts and ends there, and changing the base point requires choosing a path that conjugates loop classes.

Path homotopy is stricter than free homotopy: endpoints must remain fixed for all time. That is the condition that lets loop classes multiply and gives a well-defined group operation.

Continuous maps induce group homomorphisms on fundamental groups. The identity and composition laws are exactly the functor laws, so `pi_1` is best understood as a covariant functor from pointed topological spaces to groups.

Retractions, deformation retractions, and homotopy equivalences are not just shape language. They control induced maps on `pi_1`: retractions give injectivity/surjectivity statements, while homotopy equivalences give isomorphisms after the base-point correction is handled.

The chapter's computational theme is finite control over continuous data. Homotopies are maps on squares, path products are piecewise schedules, manifold loops can be subdivided by Lebesgue-number arguments, and category diagrams record which algebraic statement follows from which topological construction.
